# Long Term Arrears (LTA) report – SQL pipeline
Replaces `LTA+Forecast (P_DW & LU Jul-21 onwards).sas`. **Run all cells top to bottom** (VS Code: *Run All*).
The run **stops with a clear message** if the month-end data is not loaded yet, or if any built-in check fails.
The email is only ever saved as a **draft** – it is never sent.

Only one input is needed – the first day of the snapshot month (cell 1). Everything else is derived from it.

In [ ]:
# ============ 1. INPUT & SETTINGS ============
import datetime as dt
from pathlib import Path

# ---- the single date input (SAS: %let monthend='01Oct2026'd) ----
# Default = first day of the current month, as per the process note.
# If that month's snapshot is not in the DW yet, the pre-check below stops the run and tells you.
# To run an earlier snapshot type it here, e.g. dt.date(2026, 10, 1)  (must be a 1st of the month).
MONTH_START = dt.date.today().replace(day=1)

# ---- where things go ----
OUT_DIR    = Path(r"H:\Portfolio Management\Long Term Arrears\Detailed MIA Report")   # same as SAS exports
SHARED_DIR = Path(r"\\sbsfiles\Credit_Management_CreditRiskMI\Long Term Arrears")     # for the email link
REPORT_SUMMARY = OUT_DIR / "Report Summary.xlsx"            # workbook that feeds the Word report (charts live in here)

# ---- switches ----
UPDATE_REPORT_SUMMARY   = False   # True = also rotate the previous-month copy and refresh sheets inside Report Summary.xlsx
                                  # (needs Windows + Excel; see section 9). Off by default until you have tried it once.
STOP_ON_DUPLICATE_KEYS  = True    # SAS "merge by AccountNumber" silently misbehaves on duplicate account numbers; stop if found
EXPORT_ACCOUNT_LEVEL    = False   # True = write core/enriched tables to CSV (replacement for the SAS 'out' library, for account checks)
RUN_BY_SOMEONE_ELSE     = False   # True = CC Sam Hutchinson & Anis Naeem on the email (process note: "when run by someone else")

# ---- SQL Server connection (same server/database/Windows login as the SAS libname) ----
SERVER, DATABASE = r"SBSPSQLVS101\MISPSQL01", "P_DW"
ODBC_DRIVER = "SQL Server"        # same driver SAS used; use "ODBC Driver 17 for SQL Server" if you prefer

# ---- email recipients (names are resolved by Outlook; no addresses are stored here) ----
EMAIL_TO = ["Zoe Sharpe", "Sarah Anderton", "Dale McInroy", "Damian Frodsham", "CRM Quality Assurance",
            "Credit Management Seniors", "Phillip Baxendale", "Elisa Powell"]
EMAIL_CC = ["Sam Hutchinson", "Anis Naeem"]

assert MONTH_START.day == 1, "MONTH_START must be the first day of a month"
YYMM = MONTH_START.strftime("%y%m")
print("Snapshot month:", MONTH_START.strftime("%B %Y"), "| sheet suffix:", YYMM)

In [ ]:
# ============ 2. CONNECTION & HELPERS ============
import time, re, warnings
import pandas as pd
import sqlalchemy as sa
from sqlalchemy.engine import URL
warnings.filterwarnings("ignore", category=UserWarning)

class PipelineStop(Exception):
    """Raised to halt the run with a readable message (pre-check or built-in check failed)."""

def stop(msg):
    raise PipelineStop("\n\n  STOPPED: " + msg + "\n")

engine = sa.create_engine(URL.create("mssql+pyodbc", host=SERVER, database=DATABASE,
                          query={"driver": ODBC_DRIVER, "trusted_connection": "yes"}),
                          isolation_level="AUTOCOMMIT")
conn = engine.connect()          # ONE connection for the whole run: the #temp tables live on it

def run_sql(sql, params=None, label=""):
    """Run a T-SQL batch on the session connection."""
    t0 = time.time()
    conn.exec_driver_sql("SET NOCOUNT ON;\n" + sql, params or ())
    if label: print(f"  {label:<28} {time.time()-t0:6.1f}s")

def q(sql):
    """Query -> DataFrame."""
    return pd.read_sql_query(sql, conn)

def scalar(sql):
    return conn.exec_driver_sql(sql).scalar()

print("Connected to", SERVER, "/", DATABASE)

## 3. Pre-check – is the month-end data loaded?
Stops the run if the snapshot is missing/empty, or if any of the 48 history months is empty.

In [ ]:
months = q(f"""SELECT dm.CalendarMonthYear AS Yyyymm, COUNT(f.FK_Account) AS Rows_
               FROM P_DW.dbo.dimMonth dm
               LEFT JOIN P_DW.dbo.factMortgageAccountMonthSS f ON f.FK_SnapshotDate = dm.PK_Month
               WHERE dm.CalendarMonthYear BETWEEN CONVERT(VARCHAR,{(MONTH_START.year-5)*100+MONTH_START.month})
                                              AND CONVERT(VARCHAR,{MONTH_START.year*100+MONTH_START.month})
               GROUP BY dm.CalendarMonthYear""")
months["Yyyymm"] = months["Yyyymm"].astype(str)
have = dict(zip(months.Yyyymm, months.Rows_))
wanted = [(MONTH_START.year*12 + MONTH_START.month-1 - k) for k in range(48)]
wanted = [f"{m//12}{m%12+1:02d}" for m in wanted]

if have.get(wanted[0], 0) == 0:
    stop(f"Snapshot {wanted[0]} is NOT loaded in P_DW yet (factMortgageAccountMonthSS has no rows for it).\n"
         f"  Wait for the month-end load, or set MONTH_START in cell 1 to the latest loaded month "
         f"(latest loaded: {max([m for m, n in have.items() if n > 0], default='none')}).")
empty = [m for m in wanted if have.get(m, 0) == 0]
if empty:
    stop(f"These history months have no data in the fact table: {empty}. The 48-month arrears history would be incomplete.")

ods = {t: scalar(f"SELECT COUNT(*) FROM P_ODS.SKIPCORE.{t}") for t in ["ARR97", "ADM98"]}
if not all(ods.values()):
    stop(f"P_ODS.SKIPCORE tables look empty: {ods}")
print(f"Pre-check passed: snapshot {wanted[0]} has {have[wanted[0]]:,} fact rows; 48 history months present; ODS tables loaded.")

## 4. SQL pipeline
Each cell runs one stage on the same connection (the `#temp` tables live for this run only – nothing is written to the database). The SQL is plain T-SQL, so any stage can also be pasted into SSMS.

**00_params** – Parameters: the single date input is passed in here; the 48-month window is derived from it.

In [ ]:
SQL_00 = r"""
/* 00_params: single date input -> derived parameters and the 48-month window.
   The only value passed in is the first day of the snapshot month (the SAS &monthend). */
SELECT  p.MonthStart
      , CAST(YEAR(p.MonthStart)*100+MONTH(p.MonthStart) AS VARCHAR(6)) AS Yyyymm
      , RIGHT(CAST(YEAR(p.MonthStart)*100+MONTH(p.MonthStart) AS VARCHAR(6)),4) AS Yymm
      , DATEADD(MONTH,-1,p.MonthStart) AS PrevMonthStart
      , 48 AS MaxMonths /* SAS &max - do not change */
INTO    #params
FROM    (SELECT CAST(? AS DATE) AS MonthStart) p;

WITH n AS (
    SELECT 0 AS MonthOffset
    UNION ALL
    SELECT n.MonthOffset+1 FROM n CROSS JOIN #params p WHERE n.MonthOffset+1<p.MaxMonths
)
SELECT  n.MonthOffset
      , CAST(YEAR(DATEADD(MONTH,-n.MonthOffset,p.MonthStart))*100+MONTH(DATEADD(MONTH,-n.MonthOffset,p.MonthStart)) AS VARCHAR(6)) AS Yyyymm
INTO    #months
FROM    n CROSS JOIN #params p;
"""
run_sql(SQL_00, (MONTH_START,), "00_params")

**01_base** – One pass over the fact table for all 48 months (replaces SAS core + m1..m7 + the 48 monthly step-2 queries).

In [ ]:
SQL_01 = r"""
/* 01_base: ONE pass over the fact table for all 48 months (replaces SAS core + m1..m7 + 48 x step-2 queries).
   Filters are identical to the SAS ones. One row per account per month. */
SELECT  mo.MonthOffset
      , CASE WHEN lu.Allocated_AccountNumber IS NOT NULL THEN lu.Allocated_AccountNumber ELSE da.AccountNumber END AS AccountNumber
      , da.AccountStartDate
      , da.AccountEndDate
      , dm.BatchDate
      , fma.CapitalBalanceLive
      , CASE WHEN dc.CompanyName IN ('Amber Homeloans Ltd','North Yorkshire Mortgages') AND fma.StandardArrearsAmount>0 THEN fma.StandardArrearsAmount
             WHEN dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsAmount>0 THEN fma.ContractualArrearsAmount
             ELSE 0 END AS ArrAmt
      , CASE WHEN dc.CompanyName IN ('Amber Homeloans Ltd','North Yorkshire Mortgages') THEN fma.StandardArrearsInMonths
             WHEN dc.CompanyName IN ('Skipton Building Society') THEN fma.ContractualArrearsInMonths END AS MIARaw
      , CASE WHEN das.AccountStatus='Possession' THEN 1 ELSE 0 END AS Pos
      , CASE WHEN dpd.PK_Date IS NULL THEN 0 ELSE 1 END AS PossDateMatched
      , CASE WHEN dpd.CalendarYear*100+dpd.CalendarMonthNumber=dm.CalendarMonthYear THEN 1 ELSE 0 END AS RepLM
      , CASE WHEN YEAR(da.AccountEndDate)*100+MONTH(da.AccountEndDate)=dm.CalendarMonthYear THEN 1 ELSE 0 END AS RedLM
INTO    #fm
FROM    P_DW.dbo.factMortgageAccountMonthSS fma
INNER JOIN P_DW.dbo.dimAccount da         ON fma.FK_Account = da.PK_Account
INNER JOIN P_DW.dbo.dimCompany dc         ON fma.FK_Company = dc.PK_Company
INNER JOIN P_DW.dbo.dimAccountStates das  ON fma.FK_AccountStates = das.PK_AccountStates
INNER JOIN P_DW.dbo.dimProduct dp         ON fma.FK_CurrentMortgageProduct = dp.PK_Product
INNER JOIN P_DW.dbo.dimMonth dm           ON fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN #months mo                     ON dm.CalendarMonthYear = mo.Yyyymm
LEFT JOIN  P_DW.dbo.dimDate dpd           ON fma.FK_PossessionDate = dpd.PK_Date
LEFT JOIN  P_ODS.reference.AHLNYMMigratedAccounts lu
       ON  da.AccountPool = lu.AHLNYM_PoolID AND da.AccountNumber = lu.AHLNYM_AccountNumber
WHERE   dc.CompanyName IN ('Skipton Building Society','Amber Homeloans Ltd','North Yorkshire Mortgages')
AND     da.AccountPool<>'DRC00001'
AND     da.AccountSubType='Primary'
AND     da.CurrentAccountClassification<>'Pure Commercial'
AND     da.AccountStartDate IS NOT NULL /*Removes Delayed Completions*/
AND     dp.ProductCode NOT IN ('MIN01','HOM10','HM120','HM121','HM122','HM123','HM124','HM125') /*SBS deed accounts + Holmesdale deed and fair value accounts*/
AND     (das.AccountStatus IN ('Live','Possession') OR
        (da.AccountEndDate IS NOT NULL AND YEAR(da.AccountEndDate)*100+MONTH(da.AccountEndDate)=dm.CalendarMonthYear)); /*Includes month end redemptions*/

/* Month-level flags derived once (Y/N columns of SAS step 2 become 1/0) */
SELECT  f.MonthOffset, f.AccountNumber, f.Pos, f.ArrAmt
      , CASE WHEN f.CapitalBalanceLive<=0 THEN 0 ELSE f.CapitalBalanceLive END AS Bal
      , CASE WHEN f.MIARaw>=1 THEN 1 ELSE 0 END AS M1P
      , CASE WHEN f.MIARaw>=3 THEN 1 ELSE 0 END AS M3P
      , CASE WHEN f.MIARaw>=6 THEN 1 ELSE 0 END AS M6P
      , CASE WHEN f.CapitalBalanceLive>0 AND f.ArrAmt>0 AND ROUND((f.ArrAmt/f.CapitalBalanceLive)*100,6)>=0.8333 THEN 1 ELSE 0 END AS R1P
      , CASE WHEN f.CapitalBalanceLive>0 AND f.ArrAmt>0 AND ROUND((f.ArrAmt/f.CapitalBalanceLive)*100,6)>=2.4999 THEN 1 ELSE 0 END AS R3P
      , CASE WHEN f.CapitalBalanceLive>0 AND f.ArrAmt>0 AND ROUND((f.ArrAmt/f.CapitalBalanceLive)*100,6)>=4.9998 THEN 1 ELSE 0 END AS R6P
INTO    #hist
FROM    #fm f;
"""
run_sql(SQL_01, label="01_base")

**02_core** – Account history: unbroken-run lengths for each MIA / ATBR definition (replaces the 48-character SAS strings).

In [ ]:
SQL_02 = r"""
/* 02_core: SAS steps 1, 3, 4 (flags) and 5 (rating / expected arrears) at account level.
   SAS built 48 x 6 Y/N columns, concatenated them into 48-character strings and used INDEX()/SUBSTR().
   Equivalent here without strings:
     Dur     = length of the unbroken run of Y starting at the snapshot month        (= INDEX(string,'N')-1, or 48)
     DurPrev = length of the unbroken run of Y starting one month earlier             (= SUBSTR(string,2,k)='Y..Y' test for k <= DurPrev)
   A month where the account is absent counts as 'N', exactly as the SAS merge left it blank -> 'N'. */

/* Rows where the flag is Y, numbered in month order; the run is unbroken while MonthOffset = row number - 1 */
SELECT  y.AccountNumber, y.Series, y.MonthOffset
      , ROW_NUMBER() OVER (PARTITION BY y.AccountNumber, y.Series ORDER BY y.MonthOffset) AS rn
      , ROW_NUMBER() OVER (PARTITION BY y.AccountNumber, y.Series, CASE WHEN y.MonthOffset>=1 THEN 1 ELSE 0 END ORDER BY y.MonthOffset) AS rn1
INTO    #yrows
FROM   (SELECT h.AccountNumber, h.MonthOffset, s.Series
        FROM   #hist h
        CROSS APPLY (VALUES ('M1P',h.M1P),('M3P',h.M3P),('M6P',h.M6P),('R1P',h.R1P),('R3P',h.R3P),('R6P',h.R6P)) s(Series,Flag)
        WHERE  s.Flag=1) y;

SELECT  r.AccountNumber
      , COUNT(CASE WHEN r.Series='M1P' AND r.MonthOffset=r.rn-1 THEN 1 END) AS M1PDur
      , COUNT(CASE WHEN r.Series='M3P' AND r.MonthOffset=r.rn-1 THEN 1 END) AS M3PDur
      , COUNT(CASE WHEN r.Series='M6P' AND r.MonthOffset=r.rn-1 THEN 1 END) AS M6PDur
      , COUNT(CASE WHEN r.Series='R1P' AND r.MonthOffset=r.rn-1 THEN 1 END) AS R1PDur
      , COUNT(CASE WHEN r.Series='R3P' AND r.MonthOffset=r.rn-1 THEN 1 END) AS R3PDur
      , COUNT(CASE WHEN r.Series='R6P' AND r.MonthOffset=r.rn-1 THEN 1 END) AS R6PDur
      , COUNT(CASE WHEN r.Series='M1P' AND r.MonthOffset>=1 AND r.MonthOffset=r.rn1 THEN 1 END) AS M1PDurPrev
      , COUNT(CASE WHEN r.Series='M3P' AND r.MonthOffset>=1 AND r.MonthOffset=r.rn1 THEN 1 END) AS M3PDurPrev
      , COUNT(CASE WHEN r.Series='M6P' AND r.MonthOffset>=1 AND r.MonthOffset=r.rn1 THEN 1 END) AS M6PDurPrev
      , COUNT(CASE WHEN r.Series='R1P' AND r.MonthOffset>=1 AND r.MonthOffset=r.rn1 THEN 1 END) AS R1PDurPrev
      , COUNT(CASE WHEN r.Series='R3P' AND r.MonthOffset>=1 AND r.MonthOffset=r.rn1 THEN 1 END) AS R3PDurPrev
      , COUNT(CASE WHEN r.Series='R6P' AND r.MonthOffset>=1 AND r.MonthOffset=r.rn1 THEN 1 END) AS R6PDurPrev
INTO    #dur
FROM    #yrows r
GROUP BY r.AccountNumber;

/* Last 7 months' balance / arrears (SAS m1..m7) and prior-month possession flag (Pos2) */
SELECT  h.AccountNumber
      , MAX(CASE WHEN h.MonthOffset=0 THEN h.Bal END) AS Bal1, MAX(CASE WHEN h.MonthOffset=0 THEN h.ArrAmt END) AS Arr1
      , MAX(CASE WHEN h.MonthOffset=1 THEN h.Bal END) AS Bal2, MAX(CASE WHEN h.MonthOffset=1 THEN h.ArrAmt END) AS Arr2
      , MAX(CASE WHEN h.MonthOffset=2 THEN h.Bal END) AS Bal3, MAX(CASE WHEN h.MonthOffset=2 THEN h.ArrAmt END) AS Arr3
      , MAX(CASE WHEN h.MonthOffset=3 THEN h.Bal END) AS Bal4, MAX(CASE WHEN h.MonthOffset=3 THEN h.ArrAmt END) AS Arr4
      , MAX(CASE WHEN h.MonthOffset=4 THEN h.Bal END) AS Bal5, MAX(CASE WHEN h.MonthOffset=4 THEN h.ArrAmt END) AS Arr5
      , MAX(CASE WHEN h.MonthOffset=5 THEN h.Bal END) AS Bal6, MAX(CASE WHEN h.MonthOffset=5 THEN h.ArrAmt END) AS Arr6
      , MAX(CASE WHEN h.MonthOffset=6 THEN h.Bal END) AS Bal7, MAX(CASE WHEN h.MonthOffset=6 THEN h.ArrAmt END) AS Arr7
      , MAX(CASE WHEN h.MonthOffset=1 THEN h.Pos END) AS Pos2
INTO    #last7
FROM    #hist h
WHERE   h.MonthOffset<=6
GROUP BY h.AccountNumber;

/* Core: snapshot-month accounts + their history (SAS step 1 core, merged with perf and step 3 durations) */
SELECT  f.AccountNumber
      , CONVERT(DATE,f.AccountStartDate) AS CompDate
      , x.AgeDays
      , x.AgeYears
      , CASE WHEN x.AgeYears>=3 AND f.CapitalBalanceLive>0 THEN 1 ELSE 0 END AS Mature
      , CASE WHEN x.AgeYears>=3 AND f.CapitalBalanceLive>0 THEN f.CapitalBalanceLive ELSE 0 END AS MatureBal
      , CASE WHEN f.ArrAmt>0 AND f.CapitalBalanceLive>0 THEN ROUND((f.ArrAmt/f.CapitalBalanceLive)*100,6) ELSE 0 END AS ATBR
      , CASE WHEN f.MIARaw>0 THEN ROUND(f.MIARaw,3) ELSE 0 END AS MIA
      , f.Pos AS Repossessed
      , f.RepLM
      , f.RedLM
      , l.Bal1, l.Arr1, l.Bal2, l.Arr2, l.Bal3, l.Arr3, l.Bal4, l.Arr4, l.Bal5, l.Arr5, l.Bal6, l.Arr6, l.Bal7, l.Arr7
      , l.Pos2
      , COALESCE(d.M1PDur,0) AS M1PDur, COALESCE(d.M3PDur,0) AS M3PDur, COALESCE(d.M6PDur,0) AS M6PDur
      , COALESCE(d.R1PDur,0) AS R1PDur, COALESCE(d.R3PDur,0) AS R3PDur, COALESCE(d.R6PDur,0) AS R6PDur
      , COALESCE(d.M1PDurPrev,0) AS M1PDurPrev, COALESCE(d.M3PDurPrev,0) AS M3PDurPrev, COALESCE(d.M6PDurPrev,0) AS M6PDurPrev
      , COALESCE(d.R1PDurPrev,0) AS R1PDurPrev, COALESCE(d.R3PDurPrev,0) AS R3PDurPrev, COALESCE(d.R6PDurPrev,0) AS R6PDurPrev
INTO    #core_a
FROM    #fm f
CROSS APPLY (SELECT CASE WHEN f.AccountStartDate IS NULL OR f.AccountStartDate>f.BatchDate THEN 0 ELSE DATEDIFF(DAY,f.AccountStartDate,f.BatchDate) END AS AgeDays
                   , CASE WHEN f.AccountStartDate IS NULL OR f.AccountStartDate>f.BatchDate THEN 0
                          ELSE ROUND(CONVERT(FLOAT,(DATEDIFF(DAY,f.AccountStartDate,f.BatchDate)))/365,2) END AS AgeYears) x
LEFT JOIN #last7 l ON l.AccountNumber=f.AccountNumber
LEFT JOIN #dur   d ON d.AccountNumber=f.AccountNumber
WHERE   f.MonthOffset=0;
"""
run_sql(SQL_02, label="02_core")

**02b_core_flags** – Roll-in / held / roll-out flags, payment rating and expected arrears (SAS steps 4-5).

In [ ]:
SQL_02b = r"""
/* 02b: SAS step 4 account-level flags + step 5 (payment rating, average arrears reduction, expected arrears).
   SAS treats a missing number as smaller than any number in comparisons; arrears are never negative,
   so COALESCE(arrears,-1) reproduces that behaviour exactly. In MAX(0,x) a missing x counts as 0. */
SELECT  a.*
      , CASE WHEN a.M6PDur=12 AND a.Repossessed=0 THEN 1 ELSE 0 END AS MLTAIn
      , CASE WHEN a.M6PDur>12 AND a.M6PDur<=24 AND a.Repossessed=0 THEN 1 ELSE 0 END AS MLTAHeld1
      , CASE WHEN a.M6PDur>24 AND a.M6PDur<=36 AND a.Repossessed=0 THEN 1 ELSE 0 END AS MLTAHeld2
      , CASE WHEN a.M6PDur>36 AND a.Repossessed=0 THEN 1 ELSE 0 END AS MLTAHeld3
      , CASE WHEN a.M6PDurPrev>=12 AND ((a.M6PDur=0 AND a.Pos2=0) OR a.RepLM=1) THEN 1 ELSE 0 END AS MLTAOut
      , CASE WHEN a.M6PDur=11 AND a.Repossessed=0 THEN 1 ELSE 0 END AS NextIn_1M
      , CASE WHEN a.M6PDur=10 AND a.Repossessed=0 THEN 1 ELSE 0 END AS NextIn_2M
      , CASE WHEN a.M6PDur=9  AND a.Repossessed=0 THEN 1 ELSE 0 END AS NextIn_3M
      , rt.Rating
      , rt.AvArrRed
      , CASE WHEN rt.Rating='Green' OR (rt.Rating='Amber' AND rt.AvArrRed>0) THEN a.Arr1-(rt.AvArrRed*1) ELSE a.Arr1 END AS ExpAB1
      , CASE WHEN rt.Rating='Green' OR (rt.Rating='Amber' AND rt.AvArrRed>0) THEN a.Arr1-(rt.AvArrRed*2) ELSE a.Arr1 END AS ExpAB2
      , CASE WHEN rt.Rating='Green' OR (rt.Rating='Amber' AND rt.AvArrRed>0) THEN a.Arr1-(rt.AvArrRed*3) ELSE a.Arr1 END AS ExpAB3
INTO    #core
FROM    #core_a a
CROSS APPLY (SELECT COALESCE(a.Arr1,-1) AS A1, COALESCE(a.Arr2,-1) AS A2, COALESCE(a.Arr3,-1) AS A3
                  , COALESCE(a.Arr4,-1) AS A4, COALESCE(a.Arr5,-1) AS A5, COALESCE(a.Arr6,-1) AS A6, COALESCE(a.Arr7,-1) AS A7) c
CROSS APPLY (SELECT (CASE WHEN c.A1>c.A2 THEN 1 ELSE 0 END) AS Det1, (CASE WHEN c.A2>c.A3 THEN 1 ELSE 0 END) AS Det2
                  , (CASE WHEN c.A3>c.A4 THEN 1 ELSE 0 END) AS Det3, (CASE WHEN c.A4>c.A5 THEN 1 ELSE 0 END) AS Det4
                  , (CASE WHEN c.A1<c.A2 THEN 1 ELSE 0 END) AS Imp1, (CASE WHEN c.A2<c.A3 THEN 1 ELSE 0 END) AS Imp2
                  , (CASE WHEN c.A3<c.A4 THEN 1 ELSE 0 END) AS Imp3) f
CROSS APPLY (SELECT CASE WHEN a.Arr2 IS NOT NULL AND a.Arr2>a.Arr1 THEN a.Arr2-a.Arr1 ELSE 0 END AS Red1
                  , CASE WHEN a.Arr3 IS NOT NULL AND a.Arr2 IS NOT NULL AND a.Arr3>a.Arr2 THEN a.Arr3-a.Arr2 ELSE 0 END AS Red2
                  , CASE WHEN a.Arr4 IS NOT NULL AND a.Arr3 IS NOT NULL AND a.Arr4>a.Arr3 THEN a.Arr4-a.Arr3 ELSE 0 END AS Red3) rd
CROSS APPLY (SELECT CASE WHEN f.Det1=1 OR (f.Det2+f.Det3+f.Det4)>=2 THEN 'Red'
                         WHEN f.Imp1=1 AND f.Imp2=1 AND f.Imp3=1 THEN 'Green'
                         ELSE 'Amber' END AS Rating
                  /* median of three, then ROUND(...,1) in SAS = nearest whole pound */
                  , ROUND(CASE WHEN (rd.Red1>=rd.Red2 AND rd.Red1<=rd.Red3) OR (rd.Red1<=rd.Red2 AND rd.Red1>=rd.Red3) THEN rd.Red1
                               WHEN (rd.Red2>=rd.Red1 AND rd.Red2<=rd.Red3) OR (rd.Red2<=rd.Red1 AND rd.Red2>=rd.Red3) THEN rd.Red2
                               ELSE rd.Red3 END,0) AS AvArrRed) rt;
"""
run_sql(SQL_02b, label="02b_core_flags")

**03_lookups** – Snapshot-month lookups: account detail, CMS, deceased, redemption statements (SAS step 6).

In [ ]:
SQL_03 = r"""
/* 03_lookups: snapshot-month lookups (SAS step 6: accounts, CMS, Dec, redsta). Logic unchanged. */

/*Account Data*/
SELECT  CASE WHEN lu.Allocated_AccountNumber IS NOT NULL THEN lu.Allocated_AccountNumber ELSE da.AccountNumber END AS AccountNumber
      , CASE WHEN dc.CompanyName='Amber Homeloans Ltd' OR da.AccountOrigin='Amber Homeloans Ltd' THEN 'AHL'
             WHEN dc.CompanyName='North Yorkshire Mortgages' OR da.AccountOrigin='North Yorkshire Mortgages Ltd' THEN 'NYM'
             ELSE 'SBS' END AS Lender
      , da.AccountNumber AS AccountOriginal
      , da.CurrentAccountClassification
      , dsp.SubPopulationCode
      , CASE WHEN dc.CompanyName IN ('Amber Homeloans Ltd','North Yorkshire Mortgages') AND fma.StandardArrearsInMonths>0 THEN fma.StandardArrearsInMonths
             WHEN dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsInMonths>0 THEN fma.ContractualArrearsInMonths
             ELSE 0 END AS Contractual_MIA
      , CASE WHEN dc.CompanyName IN ('Amber Homeloans Ltd','North Yorkshire Mortgages') AND fma.StandardArrearsAmount>0 THEN fma.StandardArrearsAmount
             WHEN dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsAmount>0 THEN fma.ContractualArrearsAmount
             ELSE 0 END AS Contractual_Arrears
      , fma.HalifaxValuation AS HPI_Val
      , ROUND(fma.[HPI LTV],1) AS LTV
      , CASE WHEN drt.RepaymentTypeGroup='Capital & Interest' THEN 'C&I' WHEN drt.RepaymentTypeGroup='Part & Part' THEN 'P&P' ELSE 'I/O' END AS Rep_Type
      , dms.ONSRegionCode AS HPI_Region
      , YEAR(da.AccountStartDate) AS CompletionYear
      , CASE WHEN darr.ArrangementTypeCode<1 THEN 'N' ELSE 'Y' END AS ArrangementLive
      , CASE WHEN da.AccountEndDate IS NULL AND a97.dc_ins_o_dt IS NOT NULL
              AND (a97.dc_wdrwn_o_dt IS NULL OR a97.dc_wdrwn_o_dt < a97.dc_ins_o_dt)
              AND (a97.dc_poses_o_dt IS NULL OR a97.dc_poses_o_dt < a97.dc_ins_o_dt) THEN 'Y' ELSE 'N' END AS Litigation
      , CONVERT(DATE, a97.dc_evict_o_dt) AS EvictionDate
      , CONVERT(DATE, a97.dc_poses_o_dt) AS PossessionDate
      , CONVERT(DATE, a97.dc_wdrwn_o_dt) AS ProceedingsWithdrawnDate
      , CASE WHEN da.AccountEndDate IS NOT NULL AND da.AccountEndDate<dm.ReportingMonth THEN 1 ELSE 0 END AS RedeemedFlag
      , CASE WHEN das.AccountStatus='Possession' THEN 'Y' ELSE 'N' END AS Repossessed_Ind
INTO    #accounts
FROM    P_DW.dbo.factMortgageAccountMonthSS fma
INNER JOIN P_DW.dbo.dimAccount da            ON fma.FK_Account = da.PK_Account
INNER JOIN P_DW.dbo.dimCompany dc            ON fma.FK_Company = dc.PK_Company
INNER JOIN P_DW.dbo.dimRepaymentType drt     ON fma.FK_RepaymentType = drt.PK_RepaymentType
INNER JOIN P_DW.dbo.dimMortgageSecurity dms  ON fma.FK_MortgageSecurity = dms.PK_MortgageSecurity
INNER JOIN P_DW.dbo.dimProduct dp            ON fma.FK_CurrentMortgageProduct = dp.PK_Product
INNER JOIN P_DW.dbo.dimMonth dm              ON fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN P_DW.dbo.dimAccountStates das     ON fma.FK_AccountStates = das.PK_AccountStates
INNER JOIN P_DW.dbo.dimSubPopulation dsp     ON fma.FK_SubPopulation = dsp.PK_SubPopulation
INNER JOIN P_DW.dbo.dimArrangement darr      ON fma.FK_ArrangementType = darr.PK_ArrangementType
LEFT JOIN  P_ODS.reference.AHLNYMMigratedAccounts lu
       ON  da.AccountPool = lu.AHLNYM_PoolID AND da.AccountNumber = lu.AHLNYM_AccountNumber
LEFT JOIN (SELECT b.* FROM (SELECT a.*, RANK() OVER(PARTITION BY a.accnbr_o ORDER BY a.oid DESC, a._timestamp DESC) AS LatestRecord
                            FROM P_ODS.SKIPCORE.ARR97 a WHERE a.pool_id_o='SBS00001') b
           WHERE b.LatestRecord=1) a97
       ON  da.AccountNumber = a97.accnbr_o
WHERE   dc.CompanyName IN ('Skipton Building Society','Amber Homeloans Ltd','North Yorkshire Mortgages')
AND     da.AccountPool<>'DRC00001'
AND     dm.CalendarMonthYear=(SELECT Yyyymm FROM #params)
AND     da.AccountSubType='Primary'
AND     da.CurrentAccountClassification<>'Pure Commercial'
AND     da.AccountStartDate IS NOT NULL /*Removes Delayed Completions*/
AND     dp.ProductCode NOT IN ('MIN01','HOM10','HM120','HM121','HM122','HM123','HM124','HM125')
AND     (das.AccountStatus IN ('Live','Possession') OR
        (da.AccountEndDate IS NOT NULL AND YEAR(da.AccountEndDate)*100+MONTH(da.AccountEndDate)=dm.CalendarMonthYear)); /*Includes month end redemptions*/

/*Account CMS*/
SELECT  CASE WHEN lu.Allocated_PrimaryAccount IS NOT NULL THEN lu.Allocated_PrimaryAccount ELSE dpa.AccountNumber END AS AccountNumber
      , SUM(fma.AccountCMSCurrent) AS CMS_Current
      , SUM(fma.AccountCMSNextMonth) AS CMS_Next
INTO    #cms
FROM    P_DW.dbo.factMortgageAccountMonthSS fma
INNER JOIN P_DW.dbo.dimAccount da        ON fma.FK_Account = da.PK_Account
INNER JOIN P_DW.dbo.dimAccount dpa       ON fma.FK_PrimaryAccount = dpa.PK_Account
INNER JOIN P_DW.dbo.dimCompany dc        ON fma.FK_Company = dc.PK_Company
INNER JOIN P_DW.dbo.dimMonth dm          ON fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN P_DW.dbo.dimAccountStates das ON fma.FK_AccountStates = das.PK_AccountStates
LEFT JOIN  P_ODS.reference.AHLNYMMigratedAccounts lu
       ON  da.AccountPool = lu.AHLNYM_PoolID AND da.AccountNumber = lu.AHLNYM_AccountNumber
WHERE   dc.CompanyName IN ('Skipton Building Society','Amber Homeloans Ltd','North Yorkshire Mortgages')
AND     da.AccountPool<>'DRC00001'
AND     dm.CalendarMonthYear=(SELECT Yyyymm FROM #params)
AND     das.AccountStatus IN ('Live','Possession')
AND     da.CurrentAccountClassification<>'Pure Commercial'
AND     da.AccountStartDate IS NOT NULL /*Removes Delayed Completions*/
GROUP BY CASE WHEN lu.Allocated_PrimaryAccount IS NOT NULL THEN lu.Allocated_PrimaryAccount ELSE dpa.AccountNumber END;

/*Deceased */
SELECT DISTINCT
        CASE WHEN lu.Allocated_AccountNumber IS NOT NULL THEN lu.Allocated_AccountNumber ELSE da.AccountNumber END AS AccountNumber
      , 1 AS Deceased
INTO    #dec
FROM    P_DW.dbo.factMortgageAccountMonthSS fma
INNER JOIN P_DW.dbo.dimAccount da             ON fma.FK_Account = da.PK_Account
INNER JOIN P_DW.dbo.dimCompany dc             ON fma.FK_Company = dc.PK_Company
INNER JOIN P_DW.dbo.dimMonth dm               ON fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN P_DW.dbo.dimAccountStates das      ON fma.FK_AccountStates = das.PK_AccountStates
INNER JOIN P_DW.dbo.bridgeAccountCustomer bac ON da.PK_Account = bac.FK_Account
INNER JOIN P_DW.dbo.dimCustomer dcu           ON bac.FK_Customer = dcu.PK_Customer
INNER JOIN P_DW.dbo.dimMortgageSecurity dms   ON fma.FK_MortgageSecurity = dms.PK_MortgageSecurity
LEFT JOIN  P_ODS.reference.AHLNYMMigratedAccounts lu
       ON  da.AccountPool = lu.AHLNYM_PoolID AND da.AccountNumber = lu.AHLNYM_AccountNumber
WHERE   dc.CompanyName IN ('Skipton Building Society','Amber Homeloans Ltd','North Yorkshire Mortgages')
AND     da.AccountPool<>'DRC00001'
AND     dm.CalendarMonthYear=(SELECT Yyyymm FROM #params)
AND     da.AccountSubType='Primary'
AND     das.AccountStatus IN ('Live','Possession')
AND     da.CurrentAccountClassification<>'Pure Commercial'
AND     (CASE WHEN dc.CompanyName IN ('Amber Homeloans Ltd','North Yorkshire Mortgages') AND fma.StandardArrearsAmount>0 THEN fma.StandardArrearsAmount
              WHEN dc.CompanyName IN ('Skipton Building Society') AND fma.ContractualArrearsAmount>0 THEN fma.ContractualArrearsAmount
              ELSE 0 END)>0
AND     da.AccountStartDate IS NOT NULL /*Removes Delayed Completions*/
AND     (dcu.CustomerDateofDeath IS NOT NULL OR dms.PostalName LIKE ('%DECD%') OR dms.PostalName LIKE ('%PERS REPS%'));

/*Redemption statement requested in the snapshot month (SAS: previous calendar month relative to GETDATE())*/
SELECT  b.AccountNumber, b.Red_Letter_Sent, 1 AS RedStaInd
INTO    #redsta
FROM   (SELECT a.*, RANK() OVER(PARTITION BY a.AccountNumber ORDER BY a.Red_Letter_Sent DESC, a.oid DESC) AS LastSent
        FROM  (SELECT a98.accnbr_o AS AccountNumber, a98.lettcode_o AS Letter_Code
                    , CONVERT(DATE,a98.dc_creat_o_dt) AS Red_Letter_Sent, a98.oid
               FROM   P_ODS.SKIPCORE.ADM98 a98
               WHERE  a98.pool_id_o='SBS00001'
               AND    a98.lettcode_o IN ('RED001','RED01F','RED01E')
               AND    a98.dc_creat_o_dt BETWEEN (SELECT MonthStart FROM #params) AND EOMONTH((SELECT MonthStart FROM #params))) a) b
WHERE   b.LastSent=1;
"""
run_sql(SQL_03, label="03_lookups")

**04_outputs** – Summary, RAG, LTA lists and next roll-in / roll-out forecasts (SAS steps 4-7).

In [ ]:
SQL_04 = r"""
/* 04_outputs: SAS steps 4 (summary), 5 (RAG summary), 6 and 7 (account-level lists + merges).
   One enriched table is built once (core + accounts + CMS + deceased + redemption statement); every list is a selection from it. */

SELECT  c.*
      , CASE WHEN c.MLTAHeld1=1 THEN 'LT12M' WHEN c.MLTAHeld2=1 THEN '12_23M' WHEN c.MLTAHeld3=1 THEN '24M_Plus' END AS HeldGroup
      , CASE WHEN c.M6PDur-12>0 THEN c.M6PDur-12 ELSE 0 END AS LTADur
      , a.Lender, a.AccountOriginal, a.CurrentAccountClassification, a.SubPopulationCode, a.Contractual_MIA, a.Contractual_Arrears
      , a.HPI_Val, a.LTV, a.Rep_Type, a.HPI_Region, a.CompletionYear, a.ArrangementLive, a.Litigation
      , a.EvictionDate, a.PossessionDate, a.ProceedingsWithdrawnDate, a.RedeemedFlag, a.Repossessed_Ind
      , m.CMS_Current, m.CMS_Next
      , d.Deceased
      , r.Red_Letter_Sent, r.RedStaInd
      , DATEDIFF(MONTH,(SELECT MonthStart FROM #params),a.EvictionDate) AS EvMonthIdx
      , ROUND(CAST(c.ExpAB1 AS FLOAT)/NULLIF(CAST(m.CMS_Next AS FLOAT),0),5) AS ExpMIA1
      , ROUND(CAST(c.ExpAB2 AS FLOAT)/NULLIF(CAST(m.CMS_Next AS FLOAT),0),5) AS ExpMIA2
      , ROUND(CAST(c.ExpAB3 AS FLOAT)/NULLIF(CAST(m.CMS_Next AS FLOAT),0),5) AS ExpMIA3
INTO    #enriched
FROM    #core c
LEFT JOIN #accounts a ON a.AccountNumber=c.AccountNumber
LEFT JOIN #cms m      ON m.AccountNumber=c.AccountNumber
LEFT JOIN #dec d      ON d.AccountNumber=c.AccountNumber
LEFT JOIN #redsta r   ON r.AccountNumber=c.AccountNumber;

/* ---- Summary (SAS proc summary, long form: one row per metric; the notebook pivots it to the SAS column layout) ---- */
SELECT  CONCAT(v.Series, CAST(k.K AS VARCHAR(2)), x.Kind) AS Metric, SUM(x.Val) AS Value
INTO    #summary_long
FROM    #core c
CROSS APPLY (VALUES ('M1P',c.M1PDur,c.M1PDurPrev),('M3P',c.M3PDur,c.M3PDurPrev),('M6P',c.M6PDur,c.M6PDurPrev)
                  , ('R1P',c.R1PDur,c.R1PDurPrev),('R3P',c.R3PDur,c.R3PDurPrev),('R6P',c.R6PDur,c.R6PDurPrev)) v(Series,Dur,DurPrev)
CROSS JOIN (VALUES (3),(6),(12),(24)) k(K)
CROSS APPLY (VALUES ('In',   CASE WHEN v.Dur=k.K AND c.Repossessed=0 THEN 1 END)
                  , ('Held', CASE WHEN v.Dur>k.K AND c.Repossessed=0 THEN 1 END)
                  , ('Out',  CASE WHEN v.DurPrev>=k.K AND ((v.Dur=0 AND c.Pos2=0) OR c.RepLM=1) THEN -1 END)) x(Kind,Val)
GROUP BY v.Series, k.K, x.Kind;

INSERT INTO #summary_long (Metric, Value)
SELECT  z.Metric, SUM(z.Val)
FROM    #core c
CROSS APPLY (VALUES ('M',c.M6PDur,c.M6PDurPrev),('R',c.R6PDur,c.R6PDurPrev)) v(P,Dur,DurPrev)
CROSS APPLY (VALUES ('LTAIn',    CASE WHEN v.Dur=12 AND c.Repossessed=0 THEN 1 END,                CASE WHEN v.Dur=12 AND c.Repossessed=0 THEN c.Bal1 END)
                  , ('LTAHeld1', CASE WHEN v.Dur>12 AND v.Dur<=24 AND c.Repossessed=0 THEN 1 END,  CASE WHEN v.Dur>12 AND v.Dur<=24 AND c.Repossessed=0 THEN c.Bal1 END)
                  , ('LTAHeld2', CASE WHEN v.Dur>24 AND v.Dur<=36 AND c.Repossessed=0 THEN 1 END,  CASE WHEN v.Dur>24 AND v.Dur<=36 AND c.Repossessed=0 THEN c.Bal1 END)
                  , ('LTAHeld3', CASE WHEN v.Dur>36 AND c.Repossessed=0 THEN 1 END,                CASE WHEN v.Dur>36 AND c.Repossessed=0 THEN c.Bal1 END)
                  , ('LTAOut',   CASE WHEN v.DurPrev>=12 AND ((v.Dur=0 AND c.Pos2=0) OR c.RepLM=1) THEN -1 END
                               , CASE WHEN v.DurPrev>=12 AND ((v.Dur=0 AND c.Pos2=0) OR c.RepLM=1) THEN c.Bal2 END)) n(Nm,Cnt,Bal)
CROSS APPLY (VALUES (CONCAT(v.P,n.Nm), n.Cnt),(CONCAT(v.P,n.Nm,'_Bal'), n.Bal)) z(Metric,Val)
GROUP BY z.Metric;

INSERT INTO #summary_long (Metric, Value)
SELECT  t.Metric, t.Value
FROM   (SELECT 'Mature' AS Metric, SUM(Mature) AS Value FROM #core
        UNION ALL SELECT 'MatureBal', SUM(MatureBal) FROM #core
        UNION ALL SELECT 'NextIn_1M', SUM(CASE WHEN NextIn_1M=1 THEN 1 END) FROM #core
        UNION ALL SELECT 'NextIn_2M', SUM(CASE WHEN NextIn_2M=1 THEN 1 END) FROM #core
        UNION ALL SELECT 'NextIn_3M', SUM(CASE WHEN NextIn_3M=1 THEN 1 END) FROM #core
        UNION ALL SELECT '_FREQ_', COUNT(*) FROM #core) t;

/* ---- RAG rating of held LTA cases (SAS rag_rating_lta) ---- */
SELECT  e.HeldGroup
      , COUNT(*) AS _FREQ_
      , SUM(CASE WHEN e.Rating='Red'   THEN 1 END) AS Red
      , SUM(CASE WHEN e.Rating='Amber' THEN 1 END) AS Amber
      , SUM(CASE WHEN e.Rating='Green' THEN 1 END) AS Green
INTO    #rag
FROM    #enriched e
WHERE   e.MLTAHeld1+e.MLTAHeld2+e.MLTAHeld3>0
GROUP BY e.HeldGroup;

/* ---- LTA_In / LTA_Out / Held (SAS: in, out, held) in ONE table, split by ListName when exported ---- */
SELECT  x.ListName
      , e.AccountNumber, e.AgeYears, e.MIA, e.Bal1, e.Arr1, e.Rating, e.HeldGroup, e.LTADur
      , e.Lender, e.AccountOriginal, e.CurrentAccountClassification, e.SubPopulationCode, e.Contractual_MIA, e.Contractual_Arrears
      , e.HPI_Val, e.LTV, e.Rep_Type, e.HPI_Region, e.CompletionYear, e.ArrangementLive, e.Litigation
      , e.EvictionDate, e.PossessionDate, e.ProceedingsWithdrawnDate, e.RedeemedFlag, e.Repossessed_Ind
      , e.Deceased, e.Red_Letter_Sent, e.RedStaInd
INTO    #lta_lists
FROM    #enriched e
CROSS APPLY (VALUES ('LTA_In',  e.MLTAIn)
                  , ('LTA_Out', e.MLTAOut)
                  , ('Held',    CASE WHEN e.MLTAHeld1=1 OR e.MLTAHeld2=1 OR e.MLTAHeld3=1 THEN 1 ELSE 0 END)) x(ListName,Flag)
WHERE   x.Flag=1;

/* ---- Next roll-in / roll-out forecasts (SAS NextRoll_In, NextRoll_Out + group logic). Group=0 rows are kept here and
        filtered at export, exactly like SAS (out.NextRoll_* keep them, the Excel sheets do not). ---- */
SELECT  e.AccountNumber, e.AgeYears, e.MIA, e.Bal1, e.Arr1, e.M6PDur, e.Rating, e.AvArrRed, e.ExpAB1, e.ExpAB2, e.ExpAB3
      , e.Lender, e.AccountOriginal, e.SubPopulationCode, e.Contractual_MIA, e.Contractual_Arrears
      , e.HPI_Val, e.LTV, e.Rep_Type, e.HPI_Region, e.CompletionYear, e.ArrangementLive, e.Litigation
      , e.EvictionDate, e.PossessionDate, e.ProceedingsWithdrawnDate, e.RedeemedFlag, e.Repossessed_Ind
      , e.CMS_Current, e.CMS_Next, e.ExpMIA1, e.ExpMIA2, e.ExpMIA3
      , ev.EvFlag, g.Group1, g.Group2, g.Group3
      , CASE WHEN g.Group1=1 THEN 1 WHEN g.Group2=1 THEN 2 WHEN g.Group3=1 THEN 3 ELSE 0 END AS [Group]
      , e.Deceased, e.Red_Letter_Sent, e.RedStaInd
INTO    #nextroll_in
FROM    #enriched e
CROSS APPLY (SELECT CASE WHEN e.EvictionDate IS NULL THEN 0 WHEN e.EvMonthIdx BETWEEN 0 AND 2 THEN e.EvMonthIdx+1 ELSE 0 END AS EvFlag) ev
CROSS APPLY (SELECT CASE WHEN e.NextIn_1M=1 AND e.ExpMIA1>=6 AND ev.EvFlag<>1 THEN 1 ELSE 0 END AS Group1
                  , CASE WHEN e.NextIn_2M=1 AND e.ExpMIA2>=6 AND ev.EvFlag<>2 THEN 1 ELSE 0 END AS Group2
                  , CASE WHEN e.NextIn_3M=1 AND e.ExpMIA3>=6 AND ev.EvFlag<>3 THEN 1 ELSE 0 END AS Group3) g
WHERE   e.NextIn_1M=1 OR e.NextIn_2M=1 OR e.NextIn_3M=1;

SELECT  e.AccountNumber, e.AgeYears, e.MIA, e.Bal1, e.Arr1, e.M6PDur, e.Rating, e.AvArrRed, e.ExpAB1, e.ExpAB2, e.ExpAB3
      , e.Lender, e.AccountOriginal, e.SubPopulationCode, e.Contractual_MIA, e.Contractual_Arrears
      , e.HPI_Val, e.LTV, e.Rep_Type, e.HPI_Region, e.CompletionYear, e.ArrangementLive, e.Litigation
      , e.EvictionDate, e.PossessionDate, e.ProceedingsWithdrawnDate, e.RedeemedFlag, e.Repossessed_Ind
      , e.CMS_Current, e.CMS_Next, e.ExpMIA1, e.ExpMIA2, e.ExpMIA3
      , ev.EvFlag, g.Group1, g.Group2, g.Group3
      , CASE WHEN g.Group1=1 THEN 1 WHEN g.Group2=1 THEN 2 WHEN g.Group3=1 THEN 3 ELSE 0 END AS [Group]
      , e.Deceased, e.Red_Letter_Sent, e.RedStaInd
INTO    #nextroll_out
FROM    #enriched e
CROSS APPLY (SELECT CASE WHEN e.EvictionDate IS NULL THEN 0 WHEN e.EvMonthIdx BETWEEN 1 AND 3 THEN e.EvMonthIdx ELSE 0 END AS EvFlag) ev
/* SAS: a missing ExpMIA counts as smaller than 6, hence the COALESCE */
CROSS APPLY (SELECT CASE WHEN (e.AvArrRed>0 AND e.Rating IN ('Green','Amber') AND COALESCE(e.ExpMIA1,-1E9)<6) OR ev.EvFlag=1 THEN 1 ELSE 0 END AS Group1
                  , CASE WHEN (e.AvArrRed>0 AND e.Rating IN ('Green','Amber') AND COALESCE(e.ExpMIA2,-1E9)<6) OR ev.EvFlag=2 THEN 1 ELSE 0 END AS Group2
                  , CASE WHEN (e.AvArrRed>0 AND e.Rating IN ('Green','Amber') AND COALESCE(e.ExpMIA3,-1E9)<6) OR ev.EvFlag=3 THEN 1 ELSE 0 END AS Group3) g
WHERE   e.MLTAIn=1 OR e.MLTAHeld1=1 OR e.MLTAHeld2=1 OR e.MLTAHeld3=1;
"""
run_sql(SQL_04, label="04_outputs")

**05_additional** – Arrangement and vulnerability data for LTA cases (SAS step 9).

In [ ]:
SQL_05 = r"""
/* 05_additional: SAS step 9 - arrangement / vulnerability data for LTA, roll-in and held cases (AdditionalLTAData.xlsx, LTA List.xlsx).
   NB: SAS also built Out2 and NextRoll_Out2 but never used them (New = In2 + Held2 + NextRoll_In2), so they are not rebuilt. */

/* Arrangement data (SAS Arrang) */
SELECT  CASE WHEN lu.Allocated_PrimaryAccount IS NOT NULL THEN lu.Allocated_PrimaryAccount ELSE dpa.AccountNumber END AS AccountNumber
      , COUNT(da.AccountNumber) AS NoAcc
      , SUM(fma.AccountCMSCurrent) AS CMS
      , SUM(fma.AccountContractualPaymentArrangement) AS ArrAmn
      , SUM(fma.AccountPaymentThisMonth) AS Paid
INTO    #ap
FROM    P_DW.dbo.factMortgageAccountMonthSS fma
INNER JOIN P_DW.dbo.dimAccount da        ON fma.FK_Account = da.PK_Account
INNER JOIN P_DW.dbo.dimAccount dpa       ON fma.FK_PrimaryAccount = dpa.PK_Account
INNER JOIN P_DW.dbo.dimCompany dc        ON fma.FK_Company = dc.PK_Company
INNER JOIN P_DW.dbo.dimProduct dp        ON fma.FK_CurrentMortgageProduct = dp.PK_Product
INNER JOIN P_DW.dbo.dimMonth dm          ON fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN P_DW.dbo.dimAccountStates das ON fma.FK_AccountStates = das.PK_AccountStates
INNER JOIN P_DW.dbo.dimArrangement dar   ON fma.FK_ArrangementType = dar.PK_ArrangementType
INNER JOIN P_DW.dbo.dimDate darrd        ON fma.FK_DatePaymentArrangementEnd = darrd.PK_Date
LEFT JOIN  P_ODS.reference.AHLNYMMigratedAccounts lu
       ON  da.AccountPool = lu.AHLNYM_PoolID AND da.AccountNumber = lu.AHLNYM_AccountNumber
WHERE   dc.CompanyName IN ('Skipton Building Society','Amber Homeloans Ltd','North Yorkshire Mortgages')
AND     da.AccountPool<>'DRC00001'
AND     dm.CalendarMonthYear=(SELECT Yyyymm FROM #params)
AND     das.AccountStatus IN ('Live')
AND     da.CurrentAccountClassification<>'Pure Commercial'
AND     dp.ProductCode NOT IN ('MIN01','HOM10','HM120','HM121','HM122','HM123','HM124','HM125')
GROUP BY CASE WHEN lu.Allocated_PrimaryAccount IS NOT NULL THEN lu.Allocated_PrimaryAccount ELSE dpa.AccountNumber END;

/* Customers with an active vulnerability recorded (SAS CTE_AR) */
SELECT  DISTINCT da.AccountNumber, 1 AS ARFlag
INTO    #ar
FROM    P_DW.dbo.factMortgageAccount fma
INNER JOIN P_DW.dbo.dimAccount da            ON fma.FK_Account = da.PK_Account
INNER JOIN P_DW.dbo.bridgeAccountCustomer bac ON da.PK_Account = bac.FK_Account
                                             AND bac.DeLinkDate IS NULL AND bac.relationshipTypeCode=19
INNER JOIN P_DW.dbo.dimCustomer dc           ON bac.FK_Customer = dc.PK_Customer
WHERE   da.AccountPool='SBS00001'
AND     da.AccountSubType='Primary'
AND     da.CurrentAccountClassification='Residential'
AND     da.CurrentAccountStatus='Live'
AND     dc.CustomerNumber IN (SELECT DISTINCT c01.custnbr_u
                              FROM   P_ODS.SKIPCORE.DatCustomerVulnerability dcv
                              INNER JOIN P_ODS.SKIPCORE.CUS01 c01 ON dcv.myCustomer = c01.oid AND c01.fndr_id_o='SBS'
                              INNER JOIN P_ODS.SKIPCORE.DatCustomerVulnerability_allMyCategories dvac ON dcv.oid = dvac.DatCustomerVulnerability_oid
                              INNER JOIN P_ODS.SKIPCORE.StcVulnerabilityCategory dvc ON dvac.StcVulnerabilityCategory_oid = dvc.oid
                              WHERE  dcv.removalDate IS NULL);

SELECT  CASE WHEN lu.Allocated_AccountNumber IS NOT NULL THEN lu.Allocated_AccountNumber ELSE da.AccountNumber END AS AccountNumber
      , dar.ArrangementTypeCode AS ArrangementCode
      , dar.ArrangementDescription AS ArrangementDesc
      , CONVERT(DATE,darrd.CalendarDate) AS ArrangementEndDate
      , ap.NoAcc
      , ap.CMS
      , CASE WHEN dar.ArrangementTypeCode=-1 THEN 0 ELSE ap.ArrAmn END AS ArrangementAmn
      , ap.Paid
      , CASE WHEN dar.ArrangementTypeCode=5 AND ap.Paid<ap.CMS THEN 1 /*DD arrangements*/
             WHEN dar.ArrangementTypeCode IN ('63','65','68') AND ap.ArrAmn IN ('0','1') AND ap.Paid<ap.CMS THEN 1 /*DWP Arrangements*/
             WHEN dar.ArrangementTypeCode IN ('63','65','68') AND ap.ArrAmn>1 AND ap.Paid<ap.ArrAmn THEN 1 /*DWP Arrangements*/
             WHEN dar.ArrangementTypeCode=6 AND ap.Paid<ap.ArrAmn THEN 1 /*Concessions*/
             WHEN dar.ArrangementTypeCode=-1 THEN 0 /*No arrangements in place*/
             WHEN dar.ArrangementTypeCode NOT IN ('-1','5','6','63','65','68') AND ap.Paid<ap.ArrAmn AND ap.ArrAmn-ap.Paid>5 THEN 1 /*Other arrangements - Shortfalls over 5 pounds*/
             ELSE 0 END AS BrokenArrangement
      , ISNULL(ar.ARFlag,0) AS ARFlag
INTO    #arrang
FROM    P_DW.dbo.factMortgageAccountMonthSS fma
INNER JOIN P_DW.dbo.dimAccount da        ON fma.FK_Account = da.PK_Account
INNER JOIN P_DW.dbo.dimAccount dpa       ON fma.FK_PrimaryAccount = dpa.PK_Account
INNER JOIN P_DW.dbo.dimCompany dc        ON fma.FK_Company = dc.PK_Company
INNER JOIN P_DW.dbo.dimProduct dp        ON fma.FK_CurrentMortgageProduct = dp.PK_Product
INNER JOIN P_DW.dbo.dimMonth dm          ON fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN P_DW.dbo.dimAccountStates das ON fma.FK_AccountStates = das.PK_AccountStates
INNER JOIN P_DW.dbo.dimArrangement dar   ON fma.FK_ArrangementType = dar.PK_ArrangementType
INNER JOIN P_DW.dbo.dimDate darrd        ON fma.FK_DatePaymentArrangementEnd = darrd.PK_Date
INNER JOIN #ap ap                        ON da.AccountNumber = ap.AccountNumber
LEFT JOIN  #ar ar                        ON da.AccountNumber = ar.AccountNumber
LEFT JOIN  P_ODS.reference.AHLNYMMigratedAccounts lu
       ON  da.AccountPool = lu.AHLNYM_PoolID AND da.AccountNumber = lu.AHLNYM_AccountNumber
WHERE   dc.CompanyName IN ('Skipton Building Society','Amber Homeloans Ltd','North Yorkshire Mortgages')
AND     da.AccountPool<>'DRC00001'
AND     dm.CalendarMonthYear=(SELECT Yyyymm FROM #params)
AND     da.AccountSubType='Primary'
AND     das.AccountStatus IN ('Live')
AND     da.CurrentAccountClassification<>'Pure Commercial'
AND     dp.ProductCode NOT IN ('MIN01','HOM10','HM120','HM121','HM122','HM123','HM124','HM125')
AND     fma.ContractualArrearsInMonths>=1;

/* LTA-type list: roll-ins, held and next roll-ins (SAS In2 + Held2 + NextRoll_In2), joined to arrangement data */
SELECT  n.AccountNumber, n.MIA, n.Rating, n.Lender, n.ArrangementLive, n.Litigation, n.EvictionDate
      , CASE WHEN n.SubPopulationCode='BTL' THEN 'BTL' ELSE 'Resi' END AS AccType
      , n.LTAType
INTO    #new
FROM   (SELECT l.AccountNumber, l.MIA, l.Rating, l.Lender, l.ArrangementLive, l.Litigation, l.EvictionDate, l.SubPopulationCode
              , CASE WHEN l.ListName='LTA_In' THEN 'RolledIn'
                     WHEN l.HeldGroup='LT12M' THEN 'Held<12M'
                     WHEN l.HeldGroup='12_23M' THEN '12M<=Held<23M'
                     ELSE 'Held>=24M' END AS LTAType
        FROM   #lta_lists l WHERE l.ListName IN ('LTA_In','Held')
        UNION ALL
        SELECT r.AccountNumber, r.MIA, r.Rating, r.Lender, r.ArrangementLive, r.Litigation, r.EvictionDate, r.SubPopulationCode
              , CASE r.[Group] WHEN 1 THEN 'NextRollIn1M' WHEN 2 THEN 'NextRollIn2M' ELSE 'NextRollIn3M' END
        FROM   #nextroll_in r WHERE r.[Group]<>0) n;

/* AdditionalLTAData.xlsx (SAS AddLTAdata) */
SELECT  n.AccountNumber, n.MIA, n.Rating, n.Lender, n.ArrangementLive, n.Litigation, n.EvictionDate, n.AccType, n.LTAType
      , a.ArrangementCode, a.ArrangementDesc, a.ArrangementEndDate, a.NoAcc, a.CMS, a.ArrangementAmn, a.Paid, a.BrokenArrangement, a.ARFlag
INTO    #addlta
FROM    #new n
LEFT JOIN #arrang a ON a.AccountNumber=n.AccountNumber;

/* LTA List.xlsx (SAS lta) */
SELECT  n.AccountNumber, n.Lender, n.LTAType
INTO    #lta_list
FROM    #new n
WHERE   n.LTAType IN ('RolledIn','Held<12M','12M<=Held<23M','Held>=24M');
"""
run_sql(SQL_05, label="05_additional")

## 5. Built-in checks – any **STOP** failure halts the run; **WARN** is reported but the run continues
Each query counts violations, so **0 = pass**.

In [ ]:
CHECKS = [
 # (severity, name, SQL returning the number of violations)
 ("STOP","Snapshot accounts all matched to dimDate (SAS core inner-joins it)",
   "SELECT COUNT(*) FROM #fm WHERE MonthOffset=0 AND PossDateMatched=0"),
 ("STOP","All 48 history months present in the extract",
   "SELECT 48-COUNT(DISTINCT MonthOffset) FROM #fm"),
 ("STOP" if STOP_ON_DUPLICATE_KEYS else "WARN","One row per account per month (SAS MERGE BY AccountNumber needs unique keys)",
   "SELECT COUNT(*) FROM (SELECT AccountNumber, MonthOffset FROM #fm GROUP BY AccountNumber, MonthOffset HAVING COUNT(*)>1) d"),
 ("STOP" if STOP_ON_DUPLICATE_KEYS else "WARN","Lookups unique by AccountNumber (accounts, CMS, deceased, redemption statement, arrangements)",
   """SELECT (SELECT COUNT(*) FROM (SELECT AccountNumber FROM #accounts GROUP BY AccountNumber HAVING COUNT(*)>1) a)
         + (SELECT COUNT(*) FROM (SELECT AccountNumber FROM #cms      GROUP BY AccountNumber HAVING COUNT(*)>1) b)
         + (SELECT COUNT(*) FROM (SELECT AccountNumber FROM #dec      GROUP BY AccountNumber HAVING COUNT(*)>1) c)
         + (SELECT COUNT(*) FROM (SELECT AccountNumber FROM #redsta   GROUP BY AccountNumber HAVING COUNT(*)>1) d)
         + (SELECT COUNT(*) FROM (SELECT AccountNumber FROM #arrang   GROUP BY AccountNumber HAVING COUNT(*)>1) e)"""),
 ("STOP","Core rows = snapshot-month rows in the extract",
   "SELECT ABS((SELECT COUNT(*) FROM #core)-(SELECT COUNT(*) FROM #fm WHERE MonthOffset=0))"),
 ("STOP","Enriched table has no extra/missing rows vs core (joins did not multiply rows)",
   "SELECT ABS((SELECT COUNT(*) FROM #enriched)-(SELECT COUNT(*) FROM #core))"),
 ("STOP","Roll-in list rows = summary MLTAIn",
   "SELECT ABS((SELECT COUNT(*) FROM #lta_lists WHERE ListName='LTA_In')-(SELECT COALESCE(Value,0) FROM #summary_long WHERE Metric='MLTAIn'))"),
 ("STOP","Roll-out list rows = -summary MLTAOut",
   "SELECT ABS((SELECT COUNT(*) FROM #lta_lists WHERE ListName='LTA_Out')+(SELECT COALESCE(Value,0) FROM #summary_long WHERE Metric='MLTAOut'))"),
 ("STOP","Held list rows = summary MLTAHeld1+2+3",
   "SELECT ABS((SELECT COUNT(*) FROM #lta_lists WHERE ListName='Held')-(SELECT COALESCE(SUM(Value),0) FROM #summary_long WHERE Metric IN ('MLTAHeld1','MLTAHeld2','MLTAHeld3')))"),
 ("STOP","RAG counts (Red+Amber+Green) = held list rows",
   "SELECT ABS((SELECT COALESCE(SUM(COALESCE(Red,0)+COALESCE(Amber,0)+COALESCE(Green,0)),0) FROM #rag)-(SELECT COUNT(*) FROM #lta_lists WHERE ListName='Held'))"),
 ("STOP","Two independent definitions agree: MLTA (12m) = M6P12 (In and Held)",
   """SELECT ABS(COALESCE((SELECT Value FROM #summary_long WHERE Metric='MLTAIn'),0)-COALESCE((SELECT Value FROM #summary_long WHERE Metric='M6P12In'),0))
         + ABS(COALESCE((SELECT SUM(Value) FROM #summary_long WHERE Metric IN ('MLTAHeld1','MLTAHeld2','MLTAHeld3')),0)-COALESCE((SELECT Value FROM #summary_long WHERE Metric='M6P12Held'),0))"""),
 ("STOP","Summary row count (_FREQ_) = core rows",
   "SELECT ABS((SELECT Value FROM #summary_long WHERE Metric='_FREQ_')-(SELECT COUNT(*) FROM #core))"),
 ("STOP","Every core account has a rating; groups only 0-3",
   "SELECT (SELECT COUNT(*) FROM #core WHERE Rating IS NULL)+(SELECT COUNT(*) FROM #nextroll_in WHERE [Group] NOT IN (0,1,2,3))+(SELECT COUNT(*) FROM #nextroll_out WHERE [Group] NOT IN (0,1,2,3))"),
 ("STOP","Additional-data table has the same rows as the LTA-type list (left join did not multiply rows)",
   "SELECT ABS((SELECT COUNT(*) FROM #addlta)-(SELECT COUNT(*) FROM #new))"),
 ("WARN","Snapshot accounts missing from the account-detail lookup (SAS would leave these blank too)",
   "SELECT COUNT(*) FROM #core c WHERE NOT EXISTS (SELECT 1 FROM #accounts a WHERE a.AccountNumber=c.AccountNumber)"),
 ("WARN","Forecast accounts with no CMS_Next (expected MIA cannot be calculated)",
   "SELECT COUNT(*) FROM #nextroll_in WHERE CMS_Next IS NULL OR CMS_Next=0"),
 ("WARN","No redemption-statement letters found for the month (check ADM98 is loaded)",
   "SELECT CASE WHEN COUNT(*)=0 THEN 1 ELSE 0 END FROM #redsta"),
]
results = []
for sev, name, sql in CHECKS:
    bad = scalar(sql)
    ok = (bad == 0)
    results.append((sev, name, bad, "PASS" if ok else ("FAIL" if sev == "STOP" else "WARN")))
check_df = pd.DataFrame(results, columns=["Severity", "Check", "Violations", "Result"])
display(check_df) if "display" in globals() else print(check_df.to_string())
failed = check_df[(check_df.Severity == "STOP") & (check_df.Result == "FAIL")]
if len(failed):
    stop("Built-in checks failed:\n    - " + "\n    - ".join(failed.Check) + "\n  Nothing has been exported.")
print("All STOP checks passed.", int((check_df.Result == "WARN").sum()), "warning(s).")

## 6. Build the outputs (same sheets, columns and sort order as the SAS exports)

In [ ]:
# ---- SAS proc summary layout: _TYPE_, _FREQ_, then the VAR list in the SAS order ----
def _series(): return [f"{s}{k}" for s in ["M1P","M3P","M6P","R1P","R3P","R6P"] for k in (3,6,12,24)]
# Order = the proc summary VAR statement in the SAS code
METRIC_ORDER = (["Mature","MatureBal"]
    + [f"MLTA{n}" for n in ["Out","Held1","Held2","Held3","In"]] + [f"MLTA{n}_Bal" for n in ["Out","Held1","Held2","Held3","In"]]
    + [f"RLTA{n}" for n in ["Out","Held1","Held2","Held3","In"]] + [f"RLTA{n}_Bal" for n in ["Out","Held1","Held2","Held3","In"]]
    + [f"{s}{n}" for s in _series() for n in ["Out","Held","In"]]
    + ["NextIn_1M","NextIn_2M","NextIn_3M"])

long = q("SELECT Metric, Value FROM #summary_long")
vals = dict(zip(long.Metric, long.Value))
summary_df = pd.DataFrame([{"_TYPE_": 0, "_FREQ_": vals["_FREQ_"], **{m: vals.get(m) for m in METRIC_ORDER}}])

rag_df = q("SELECT HeldGroup, 1 AS _TYPE_, _FREQ_, Red, Amber, Green FROM #rag ORDER BY HeldGroup")

LIST_SORT = {"LTA_In": "Bal1 DESC, AccountNumber", "LTA_Out": "Bal1 DESC, AccountNumber", "Held": "LTV DESC, AccountNumber"}
lists = {n: q(f"SELECT * FROM #lta_lists WHERE ListName='{n}' ORDER BY {s}").drop(columns="ListName") for n, s in LIST_SORT.items()}
nextroll = {n: q(f"SELECT * FROM #nextroll_{n} WHERE [Group]<>0 ORDER BY [Group], Bal1 DESC, AccountNumber").rename(columns={"Group": "Group"})
            for n in ["in", "out"]}

sheets = {  # sheet name -> DataFrame (same names as the SAS proc exports)
    YYMM: summary_df, f"RAG{YYMM}": rag_df,
    "LTA_In": lists["LTA_In"], "LTA_Out": lists["LTA_Out"], "Held": lists["Held"],
    "NextRoll_In": nextroll["in"], "NextRoll_Out": nextroll["out"]}
addl_df = q("SELECT * FROM #addlta ORDER BY AccountNumber")
ltalist_df = q("SELECT * FROM #lta_list ORDER BY AccountNumber")
print({k: len(v) for k, v in sheets.items()}, "| AdditionalLTAData:", len(addl_df), "| LTA List:", len(ltalist_df))

## 7. Excel output – every data sheet has a matching `Doc_` sheet

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

RUN_STAMP = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
SNAP = MONTH_START.strftime("%B %Y")
COLS = {  # column dictionary (descriptions written from the SAS logic)
 "AccountNumber": "Account number; for migrated AHL/NYM accounts the allocated (SBS) number from P_ODS.reference.AHLNYMMigratedAccounts",
 "AgeYears": "Years from account start to snapshot BatchDate, rounded to 2dp (0 if start is after BatchDate)",
 "MIA": "Months in arrears at snapshot (contractual for SBS, standard for AHL/NYM), rounded to 3dp; 0 if not in arrears",
 "Bal1": "Capital balance at snapshot month (negative balances shown as 0)", "Arr1": "Arrears amount at snapshot month",
 "Rating": "Payment RAG from last 4-7 months' arrears: Red = rising now or at least 2 rises in prior 3 months; Green = falling 3 months running; else Amber",
 "HeldGroup": "How long a held LTA case has been LTA: LT12M, 12_23M, 24M_Plus (blank for roll-ins/roll-outs)",
 "LTADur": "Months beyond the 12-month threshold that the case has been in 6+ MIA (M6PDur-12, min 0)",
 "Lender": "AHL / NYM / SBS", "AccountOriginal": "Account number before migration mapping",
 "CurrentAccountClassification": "Residential / BTL etc. from dimAccount", "SubPopulationCode": "Sub-population (BTL or other)",
 "Contractual_MIA": "Unrounded MIA at snapshot", "Contractual_Arrears": "Arrears amount at snapshot",
 "HPI_Val": "Indexed valuation", "LTV": "HPI LTV rounded to 1dp (Held sheet is sorted by this, descending)",
 "Rep_Type": "C&I / P&P / I/O", "HPI_Region": "ONS region code", "CompletionYear": "Year of account start",
 "ArrangementLive": "Y if an arrangement type code of 1 or more is in place", "Litigation": "Y if proceedings issued and not withdrawn/possessed (ARR97)",
 "EvictionDate": "Eviction date from ARR97", "PossessionDate": "Possession date from ARR97", "ProceedingsWithdrawnDate": "Proceedings withdrawn date from ARR97",
 "RedeemedFlag": "1 if account ended before the snapshot month", "Repossessed_Ind": "Y if account status is Possession",
 "Deceased": "1 if a customer date of death exists or PostalName contains DECD / PERS REPS (arrears accounts only)",
 "Red_Letter_Sent": "Latest redemption statement letter date in the snapshot month (RED001/RED01F/RED01E)", "RedStaInd": "1 if a redemption statement was requested",
 "M6PDur": "Consecutive months in 6+ MIA up to the snapshot", "AvArrRed": "Median monthly arrears reduction over last 3 months (whole pounds)",
 "ExpAB1": "Expected arrears balance in 1 month (Green, or Amber with reduction > 0, else unchanged)", "ExpAB2": "... in 2 months", "ExpAB3": "... in 3 months",
 "CMS_Current": "Contractual monthly payment this month (summed over linked accounts)", "CMS_Next": "Contractual monthly payment next month",
 "ExpMIA1": "ExpAB1 / CMS_Next", "ExpMIA2": "ExpAB2 / CMS_Next", "ExpMIA3": "ExpAB3 / CMS_Next",
 "EvFlag": "1/2/3 if eviction falls in the month the forecast refers to (roll-in: months 0-2; roll-out: months 1-3), else 0",
 "Group1": "Qualifies for roll-in/out within 1 month", "Group2": "... 2 months", "Group3": "... 3 months",
 "Group": "Earliest month group (1-3) the case qualifies for; rows with 0 are removed from the sheet",
 "AccType": "BTL or Resi", "LTAType": "RolledIn, Held<12M, 12M<=Held<23M, Held>=24M, NextRollIn1M/2M/3M",
 "ArrangementCode": "Arrangement type code", "ArrangementDesc": "Arrangement description", "ArrangementEndDate": "Arrangement end date",
 "NoAcc": "Number of accounts under the primary account", "CMS": "Contractual monthly payment (all linked accounts)", "ArrangementAmn": "Arrangement amount (0 if none)",
 "Paid": "Payments this month", "BrokenArrangement": "1 if payments fell short of the arrangement (rules per arrangement type, shortfall > 5 pounds for 'other')",
 "ARFlag": "1 if a customer on the account has an active vulnerability recorded", "_TYPE_": "SAS proc summary column (kept so the layout matches)",
 "_FREQ_": "Number of accounts summarised", "Red": "Held cases rated Red", "Amber": "Held cases rated Amber", "Green": "Held cases rated Green",
}
DOCS = {
 YYMM: ("Summary of movements for the snapshot month (SAS: dataset 'summary' / proc summary)", "One row of totals over all snapshot accounts. Naming: <M|R><LTA or xP><threshold><Out|Held|In>; M = months-in-arrears definition, R = arrears-to-balance-ratio definition; _Bal = balance. Out is negative (-1 per case). Blank = no cases."),
 f"RAG{YYMM}": ("RAG rating of held LTA cases by time held (SAS: rag_rating_lta)", "One row per HeldGroup for cases that are held in LTA (MLTAHeld1/2/3)."),
 "LTA_In": ("Accounts that rolled INTO LTA this month (6+ MIA for exactly 12 months, not repossessed) (SAS: in)", "Sorted by Bal1 descending."),
 "LTA_Out": ("Accounts that rolled OUT of LTA this month (SAS: out)", "6+ MIA in the previous 12 months and now cured/redeemed (or repossessed in month). Sorted by Bal1 descending."),
 "Held": ("Accounts already in LTA and still there (SAS: held)", "6+ MIA for more than 12 months, not repossessed. Sorted by LTV descending."),
 "NextRoll_In": ("Forecast roll-ins in the next 1-3 months (SAS: NextRoll_In, Group<>0)", "Cases 9-11 months into 6+ MIA whose expected MIA stays >= 6 and that have no eviction in the roll month. Sorted by Group, Bal1 descending."),
 "NextRoll_Out": ("Forecast roll-outs in the next 1-3 months (SAS: NextRoll_Out, Group<>0)", "Current roll-ins and held cases that are expected to drop below 6 MIA (improving Green/Amber) or are evicted. Sorted by Group, Bal1 descending."),
 "AdditionalLTAData": ("Arrangement, performance and vulnerability data for roll-in, held and next roll-in cases (SAS: AddLTAdata)", "Sorted by AccountNumber. Use a pivot on ArrangementDesc x LTAType (excluding NextRollIn types) for the email narrative."),
 "LTA List": ("List of LTA cases (roll-in and held) with lender (SAS: lta)", "Sorted by AccountNumber."),
}

def style_header(ws):
    for c in ws[1]:
        c.font = Font(bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor="305496")
    ws.freeze_panes = "B2"

def write_book(path, frames, doc_keys=None):
    """frames: {sheet: DataFrame}. Writes each frame + a Doc_ sheet + a README index; then re-reads the file to verify row counts."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl", date_format="yyyy-mm-dd", datetime_format="yyyy-mm-dd") as xw:
        idx = []
        for name, df in frames.items():
            df = df.copy()
            for c in df.columns:
                if df[c].dtype == object and df[c].map(lambda v: isinstance(v, (dt.date, dt.datetime)) or v is None).all() and df[c].notna().any():
                    df[c] = pd.to_datetime(df[c])
            df.to_excel(xw, sheet_name=name, index=False)
            purpose, notes = DOCS.get(name, ("", ""))
            rows = [("Sheet", name), ("Purpose", purpose), ("Snapshot month", SNAP), ("Generated", RUN_STAMP),
                    ("Rows", len(df)), ("Notes", notes), ("Source", "SQL pipeline (LTA notebook); P_DW and P_ODS, Windows login"), ("", ""), ("Column", "Description")]
            rows += [(c, COLS.get(c, "")) for c in df.columns]
            pd.DataFrame(rows).to_excel(xw, sheet_name=f"Doc_{name}"[:31], index=False, header=False)
            idx.append((name, f"Doc_{name}"[:31], purpose, len(df)))
        pd.DataFrame(idx, columns=["Sheet", "Documentation sheet", "Purpose", "Rows"]).to_excel(xw, sheet_name="README", index=False)
    wb = load_workbook(path)
    for name in frames:
        style_header(wb[name])
        for i, col in enumerate(frames[name].columns, 1):
            wb[name].column_dimensions[get_column_letter(i)].width = max(10, min(28, len(str(col)) + 3))
        d = wb[f"Doc_{name}"[:31]]; d.column_dimensions["A"].width = 30; d.column_dimensions["B"].width = 110
        for r in d.iter_rows(min_row=1, max_row=9):
            r[0].font = Font(bold=True)
    wb["README"].column_dimensions["C"].width = 100
    wb.move_sheet("README", offset=-len(wb.sheetnames) + 1)
    wb.save(path)
    # verification: re-open and compare row counts with what we meant to write
    wb = load_workbook(path, read_only=True)
    for name, df in frames.items():
        n = wb[name].max_row - 1
        if n != len(df):
            stop(f"Excel verification failed for sheet {name}: file has {n} rows, expected {len(df)}")
    return path

main_path  = write_book(OUT_DIR / f"LTA Pipeline Output ({YYMM}).xlsx", sheets)
addl_path  = write_book(OUT_DIR / f"AdditionalLTAData({YYMM}).xlsx", {"AdditionalLTAData": addl_df})
lista_path = write_book(OUT_DIR / "LTA List.xlsx", {"LTA List": ltalist_df})
print("Written:\n ", main_path, "\n ", addl_path, "\n ", lista_path)

if EXPORT_ACCOUNT_LEVEL:
    d = OUT_DIR / f"AccountLevel_{YYMM}"; d.mkdir(exist_ok=True)
    for t in ["core", "enriched", "nextroll_in", "nextroll_out"]:
        q(f"SELECT * FROM #{t}").to_csv(d / f"{t}.csv", index=False)
    print("Account-level files in", d)

## 8. Self-check totals (compare these with SAS – no account data needed)

In [ ]:
tot = []
for name in ["LTA_In", "LTA_Out", "Held"]:
    d = sheets[name]; tot.append((name, len(d), round(float(d.Bal1.sum()), 2), round(float(d.Arr1.sum()), 2)))
for name in ["NextRoll_In", "NextRoll_Out"]:
    d = sheets[name]; tot.append((name, len(d), round(float(d.Bal1.sum()), 2), round(float(d.Arr1.sum()), 2)))
tot.append(("AdditionalLTAData", len(addl_df), None, None)); tot.append(("LTA List", len(ltalist_df), None, None))
selfcheck = pd.DataFrame(tot, columns=["Sheet", "Rows", "Sum Bal1", "Sum Arr1"])
print(selfcheck.to_string(index=False))
print("\nSummary sheet key figures (MIA definition: in / held / out | ATBR definition: in / held / out)")
g = lambda k: vals.get(k) or 0
print(" M:", g("MLTAIn"), g("MLTAHeld1") + g("MLTAHeld2") + g("MLTAHeld3"), g("MLTAOut"),
      "| R:", g("RLTAIn"), g("RLTAHeld1") + g("RLTAHeld2") + g("RLTAHeld3"), g("RLTAOut"))
print("Rating split of held cases:\n", rag_df.to_string(index=False))

## 9. Optional – refresh `Report Summary.xlsx` (keeps its charts)
Off by default (`UPDATE_REPORT_SUMMARY`). Windows + Excel + `pywin32` only. It follows process-note step 2, then
overwrites the *contents* of the sheets in place (sheets are never deleted, so chart links are not broken).

In [ ]:
PM_COPY = OUT_DIR / "Report Summary - Prev Month.xlsx"   # NB the process note calls this file both "Prev Month" and "PM" - confirm name

def update_report_summary():
    import shutil
    import win32com.client as win32
    if not REPORT_SUMMARY.exists():
        stop(f"{REPORT_SUMMARY} not found")
    if PM_COPY.exists(): PM_COPY.unlink()
    shutil.copy2(REPORT_SUMMARY, PM_COPY)                       # step 2: last month's file kept for forecast-vs-actual
    xl = win32.DispatchEx("Excel.Application"); xl.Visible = False; xl.DisplayAlerts = False
    try:
        wb = xl.Workbooks.Open(str(REPORT_SUMMARY))
        existing = {s.Name: s for s in wb.Worksheets}
        for name, df in sheets.items():
            ws = existing.get(name) or wb.Worksheets.Add(After=wb.Worksheets(wb.Worksheets.Count))
            ws.Name = name
            ws.UsedRange.ClearContents()
            data = [list(df.columns)] + [[None if pd.isna(v) else (v.to_pydatetime() if hasattr(v, "to_pydatetime") else v) for v in row]
                                         for row in df.itertuples(index=False, name=None)]
            ws.Range(ws.Cells(1, 1), ws.Cells(len(data), len(df.columns))).Value = data
        wb.Save(); wb.Close()
    finally:
        xl.Quit()
    print("Report Summary.xlsx refreshed; previous month copy saved as", PM_COPY.name)

if UPDATE_REPORT_SUMMARY:
    update_report_summary()
else:
    print("Skipped (UPDATE_REPORT_SUMMARY = False). Copy the sheets from", main_path.name, "into Report Summary.xlsx as before.")

## 10. Outlook draft email (saved as a draft – never sent)
Edit the wording here if you like. Recipients are resolved by Outlook from names; unresolved names are listed so you can fix them in the draft.

In [ ]:
import html, urllib.parse
nxt = (MONTH_START.replace(day=28) + dt.timedelta(days=4)).replace(day=1)          # report is named for the following month
REPORT_PDF = f"LTA Report {nxt:%y%m} ({YYMM} data).pdf"
ADDL_XLSX  = f"AdditionalLTAData({YYMM}).xlsx"
link = "file:" + urllib.parse.quote(str(SHARED_DIR).replace("\\", "/"), safe="/:")
subject = f"LTA Report {MONTH_START:%B %Y} – now available"
body = f"""<p>Morning all,</p>
<p>A quick reminder that the {MONTH_START:%B %Y} Long Term Arrears (LTA) report and accompanying LTA data file are now available here:</p>
<p><a href="{link}">{html.escape(str(SHARED_DIR))}</a></p>
<p><b>What it covers:</b> LTA volumes and balances at {MONTH_START:%B %Y} month end (MIA and ATBR definitions), accounts rolling in and out,
the 3-month forecast of expected roll-ins and roll-outs, the LTA case profile (forbearance, arrangements, vulnerability) and the pipeline.</p>
<p><b>Files:</b> {html.escape(REPORT_PDF)} and {html.escape(ADDL_XLSX)}.</p>
<p><b>Action:</b> please take a look when you have a moment and let me know if you have any questions or spot anything that needs checking.</p>
<p>Many thanks,</p>"""

missing = [f for f in (REPORT_PDF, ADDL_XLSX) if SHARED_DIR.exists() and not (SHARED_DIR / f).exists()]
if missing: print("WARNING - not yet in the shared folder (copy them there before sending):", missing)

def create_outlook_draft():
    import win32com.client as win32
    ol = win32.Dispatch("Outlook.Application")
    mail = ol.CreateItem(0)
    mail.Subject = subject
    signature = mail.HTMLBody                                   # keeps the default signature
    mail.HTMLBody = body + signature
    for name in EMAIL_TO: mail.Recipients.Add(name).Type = 1
    if RUN_BY_SOMEONE_ELSE:
        for name in EMAIL_CC: mail.Recipients.Add(name).Type = 2
    mail.Recipients.ResolveAll()
    unresolved = [r.Name for r in mail.Recipients if not r.Resolved]
    mail.Save()                                                 # into Drafts - NEVER mail.Send()
    mail.Display(False)
    return unresolved

try:
    unresolved = create_outlook_draft()
    print("Outlook draft created (not sent).", "Unresolved recipients - fix in the draft:" if unresolved else "All recipients resolved.", unresolved or "")
except Exception as e:                                          # no Outlook (e.g. not Windows): write the draft to a file instead
    fallback = OUT_DIR / f"LTA email draft ({YYMM}).html"
    fallback.write_text(f"<p><b>To:</b> {'; '.join(EMAIL_TO)}<br><b>CC:</b> {'; '.join(EMAIL_CC) if RUN_BY_SOMEONE_ELSE else ''}<br>"
                        f"<b>Subject:</b> {subject}</p>{body}", encoding="utf-8")
    print("Outlook not available (", type(e).__name__, ") - draft written to", fallback)

conn.close()
print("\nDone.")